## Distributed Tracing Fundamentals

## Introduction: What Is Distributed Tracing and Why Use It?

Welcome to the lesson on distributed tracing with **OpenTelemetry**. So far in this course, you have learned how to keep your GCP credentials secure, protect your APIs, and monitor your applications with **Cloud Monitoring**. Now, we will focus on understanding how requests move through your application, especially when it uses multiple GCP services.

**Distributed tracing** helps you see the path a request takes as it travels through different parts of your system. This is important because modern cloud applications often use many services, and it can be hard to know where problems or slowdowns happen. **OpenTelemetry** is an open-source standard that helps you trace these requests, find bottlenecks, and understand how your services work together.

### How the Tracing Flow Works



1.  **Your Python Application**: The OpenTelemetry SDK captures traces.
2.  **Spans**: These represent individual units of work (e.g., a Firestore call).
3.  **Span Exporter**: Traces are sent to a destination like the Console, OTLP, or Cloud Trace.

In this lesson, we'll use `ConsoleSpanExporter` to print traces directly to your terminal. This makes it easy to see what's happening as you learn.

---

## Recall: Using GCP Client Libraries in Python

Before we dive into tracing, let's quickly remind ourselves how we use GCP client libraries in Python. You previously used libraries like `google-cloud-firestore` to interact with GCP services.

```python
from google.cloud import firestore

db = firestore.Client()
orders_collection = db.collection('orders')
```

In this lesson, we will build on this by adding tracing to these operations.

---

## Setting Up OpenTelemetry Tracing

To use distributed tracing, you need to install the SDK. While pre-installed on CodeSignal, you would normally use:

```bash
pip install opentelemetry-api opentelemetry-sdk opentelemetry-instrumentation-grpc google-cloud-firestore
```

### Initial Configuration

Here is how you initialize the tracer to print JSON to your terminal:

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.resources import Resource

# Configure provider with service metadata
provider = TracerProvider(resource=Resource.create({"service.name": "orders-worker"}))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("OrdersWorker")

# Export spans to stdout (Console)
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))
```

*   **`TracerProvider`**: The main object managing tracing.
*   **`Resource`**: Adds metadata (like the service name) to all traces.
*   **`ConsoleSpanExporter`**: Perfect for learning; prints traces as JSON.
*   **`SimpleSpanProcessor`**: Processes each span immediately.

### Auto-Instrumentation
To trace GCP services automatically, instrument the gRPC client:

```python
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient

GrpcInstrumentorClient().instrument()
```

---

## Understanding Span Processors

A **span processor** collects completed spans and sends them to an exporter. Choosing the right one is vital for performance.

| Processor Type | Use Case | Benefit |
| :--- | :--- | :--- |
| **SimpleSpanProcessor** | Learning, debugging | Immediate visibility; simple setup. |
| **BatchSpanProcessor** | Production | High performance; lower overhead. |

> **Note:** `SimpleSpanProcessor` is synchronous and can slow down your app because it waits for the export to finish. **Always use `BatchSpanProcessor` in production.**

---

## Understanding Span Exporters

Exporters determine where your data goes:

*   **`ConsoleSpanExporter`**: Prints to your terminal. Ideal for learning.
*   **`OTLPSpanExporter`**: Sends data to a local OpenTelemetry Collector (port 4317). Ideal for local dev with visualization tools like Jaeger.
*   **`CloudTraceSpanExporter`**: Sends data directly to **Google Cloud Trace**. Ideal for production.

---

## Adding Tracing to Your Code: Step-by-Step

### 1. Configure OpenTelemetry
Set up your provider and exporter (as shown in the configuration section above).

### 2. Instrument and Connect
**Important:** Instrument gRPC *before* creating your GCP clients.

```python
GrpcInstrumentorClient().instrument()
db = firestore.Client()
```

### 3. Add a Traced Operation
Use a context manager to define a custom span.

```python
def work():
    with tracer.start_as_current_span("firestore-write"):
        db.collection("orders").document("o-200").set({"order_id": "o-200", "amount": 77.7})
```

### 4. Cleanup
Ensure all spans are flushed before the program exits.

```python
if __name__ == "__main__":
    work()
    provider.shutdown()
    print("Done")
```

---

## Moving to Production

When moving from the console to Google Cloud Trace, you only need to swap the exporter and processor:

```bash
pip install opentelemetry-exporter-gcp-trace
```

```python
from opentelemetry.exporter.cloud_trace import CloudTraceSpanExporter
from opentelemetry.sdk.trace.export import BatchSpanProcessor

cloud_trace_exporter = CloudTraceSpanExporter()
provider.add_span_processor(BatchSpanProcessor(cloud_trace_exporter))
```

This flexibility allows you to change *where* data goes without changing *how* you instrumented your code.

---

## Review and What's Next
You now know how to:
1.  **Set up** the OpenTelemetry SDK.
2.  **Instrument** gRPC for GCP services.
3.  **Create custom spans** for specific logic.
4.  **Choose** between Simple and Batch processors.
5.  **Export** to Console, OTLP, or Cloud Trace.

**Congratulations on completing the course!** You are now equipped to build secure, observable applications on Google Cloud.

## Fixing the Tracing Configuration

You have a Python script that already imports the OpenTelemetry modules and has Google Cloud Firestore code ready to run, but it is missing one key piece.

Your task is to configure the OpenTelemetry tracer provider with the service name orders-worker and set up the console exporter to print trace data to your terminal. This configuration tells the tracing system how to identify your application and where to send the trace data.

Look for the TODO comment that shows you exactly where to add the configuration lines. Remember, this setup needs to happen after the imports but before the Firestore client is created.

Once you add the missing configuration, your application will print trace data in JSON format to your terminal, allowing you to see exactly what operations are being traced and how long they take.

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import ConsoleSpanExporter
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient
from google.cloud import firestore

# TODO: Configure the tracer provider with service name 'orders-worker',
# set up the tracer with name 'OrdersWorker', create a ConsoleSpanExporter,
# and add it to the provider using SimpleSpanProcessor

# Instrument gRPC before Firestore client
GrpcInstrumentorClient().instrument()
db = firestore.Client()

def work():
    with tracer.start_as_current_span("firestore-write"):
        db.collection("orders").document("o-200").set({"order_id": "o-200", "amount": 77.7})

if __name__ == "__main__":
    work()
    provider.shutdown()
    print("Done")

```

To fix the script, you need to initialize the `TracerProvider`, define the `Resource` for your service metadata, and link the `ConsoleSpanExporter` via a `SimpleSpanProcessor`.

Here is the completed code:

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import ConsoleSpanExporter
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient
from google.cloud import firestore

# --- TODO FIXED ---
# 1. Configure the tracer provider with service name 'orders-worker'
resource = Resource.create({"service.name": "orders-worker"})
provider = TracerProvider(resource=resource)
trace.set_tracer_provider(provider)

# 2. Set up the tracer with name 'OrdersWorker'
tracer = trace.get_tracer("OrdersWorker")

# 3. Create a ConsoleSpanExporter and add it to the provider using SimpleSpanProcessor
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))
# ------------------

# Instrument gRPC before Firestore client
GrpcInstrumentorClient().instrument()
db = firestore.Client()

def work():
    with tracer.start_as_current_span("firestore-write"):
        db.collection("orders").document("o-200").set({"order_id": "o-200", "amount": 77.7})

if __name__ == "__main__":
    work()
    provider.shutdown()
    print("Done")
```

### Quick Breakdown of the Steps:
*   **`Resource.create`**: Attaches "labels" to your traces. In a microservices environment, this is how you distinguish between your "orders-worker" and a "payment-processor."
*   **`trace.set_tracer_provider(provider)`**: This sets the global tracer. Without this, the rest of your application won't know which provider to use when you call `trace.get_tracer`.
*   **`SimpleSpanProcessor`**: We use this for local debugging because it sends the data to your console immediately. In production, you'd swap this for a `BatchSpanProcessor` to save on performance.

## Enable Automatic Tracing with OpenTelemetry Instrumentation

You have set up your tracing configuration, but currently, only the operations you manually wrap in spans are being traced. The tracing library can do more — it can automatically trace supported libraries (like the Google Cloud Firestore client) so that all your database operations are traced without manual instrumentation.

Your task is to enable this automatic tracing by adding the GrpcInstrumentorClient().instrument() call from the OpenTelemetry gRPC instrumentation package. This single line tells OpenTelemetry to automatically instrument gRPC calls, which are used by the Firestore client, so all your Firestore operations will be traced automatically.

Look for the TODO comment that shows you where to add this line. The instrumentation should happen after your tracer is configured but before you create the Firestore client.

Once you add this function call, you'll see much richer trace data that includes both your manual spans and automatic traces of every Firestore operation your code performs.

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import ConsoleSpanExporter
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient
from google.cloud import firestore

# Configure provider with resource info
provider = TracerProvider(resource=Resource.create({"service.name": "orders-worker"}))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("OrdersWorker")

# Export spans to stdout (prints JSON to console)
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))

# TODO: Add the GrpcInstrumentorClient().instrument() function call to enable automatic tracing of gRPC calls

db = firestore.Client()

def work():
    with tracer.start_as_current_span("firestore-write"):
        db.collection("orders").document("o-200").set({"order_id": "o-200", "amount": 77.7})

if __name__ == "__main__":
    work()
    provider.shutdown()
    print("Done")

```

To enable automatic tracing, you need to call the instrumentation method. This "hooks" into the underlying gRPC calls that the Firestore library uses, allowing OpenTelemetry to see every request and response without you having to write manual `with` statements for every database call.

Here is the updated code:

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import ConsoleSpanExporter
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient
from google.cloud import firestore

# Configure provider with resource info
provider = TracerProvider(resource=Resource.create({"service.name": "orders-worker"}))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("OrdersWorker")

# Export spans to stdout (prints JSON to console)
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))

# --- TODO FIXED ---
# Enable automatic tracing of gRPC calls
GrpcInstrumentorClient().instrument()
# ------------------

db = firestore.Client()

def work():
    with tracer.start_as_current_span("firestore-write"):
        db.collection("orders").document("o-200").set({"order_id": "o-200", "amount": 77.7})

if __name__ == "__main__":
    work()
    provider.shutdown()
    print("Done")
```



### Why the order matters:
It is critical to call `.instrument()` **before** you initialize the `firestore.Client()`. 

When the Firestore client is created, it sets up its internal gRPC channels. If the instrumentor has already run, it "wraps" those channels with tracing logic. If you initialize the client first and *then* instrument, the client will keep using the standard, untraced channels it already created, and you won't see the automatic trace data in your console.

## Add a Custom Span to Trace a Datastore Write Operation

Automatic tracing is now set up for your Firestore operations! To gain even deeper insights, let's create a custom span to focus on a specific part of your code.

While automatic tracing captures calls to Google Cloud services, sometimes you want to highlight a particular operation within your application. Custom spans let you isolate and measure the performance of these critical steps.

Your task is to wrap the Firestore set operation with a custom span named 'firestore-write'. This will create a dedicated trace section that shows the performance of just this database write operation, making it easier to identify bottlenecks.

Look for the # TODO comment in the work() function, and add the span context manager around the db.collection("orders").document("o-200").set(...) call. This will help you see exactly how long the write operation takes in your trace output.


```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import ConsoleSpanExporter
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient
from google.cloud import firestore

# Configure provider with resource info
provider = TracerProvider(resource=Resource.create({"service.name": "orders-worker"}))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("OrdersWorker")

# Export spans to stdout (prints JSON to console)
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))

# Instrument gRPC before Firestore client
GrpcInstrumentorClient().instrument()
db = firestore.Client()

def work():
    # TODO: Wrap the Firestore set operation below with a span named 'firestore-write'
    db.collection("orders").document("o-200").set({"order_id": "o-200", "amount": 77.7})

if __name__ == "__main__":
    work()
    provider.shutdown()
    print("Done")

```

To wrap a specific block of code in a custom span, you use the `tracer.start_as_current_span()` context manager. This creates a "parent" span for the logic inside the block. Because you previously enabled the `GrpcInstrumentorClient`, any gRPC calls made inside this block will automatically become "child" spans of your custom `firestore-write` span.



Here is the updated `work()` function:

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import ConsoleSpanExporter
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient
from google.cloud import firestore

# Configure provider with resource info
provider = TracerProvider(resource=Resource.create({"service.name": "orders-worker"}))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("OrdersWorker")

# Export spans to stdout (prints JSON to console)
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))

# Instrument gRPC before Firestore client
GrpcInstrumentorClient().instrument()
db = firestore.Client()

def work():
    # --- TODO FIXED ---
    # Wrap the Firestore set operation with a span named 'firestore-write'
    with tracer.start_as_current_span("firestore-write"):
        db.collection("orders").document("o-200").set({"order_id": "o-200", "amount": 77.7})
    # ------------------

if __name__ == "__main__":
    work()
    provider.shutdown()
    print("Done")
```

### Why use Custom Spans?
While automatic instrumentation is great for seeing *that* a database was called, custom spans provide **context**:
*   **Naming**: Instead of just seeing a generic `google.firestore.v1.Firestore/Write` gRPC call, you see a span named `firestore-write` (or `process-checkout-payment`), which makes the logs much easier for humans to read.
*   **Granularity**: If you had multiple Firestore calls in one function, custom spans would let you group them together to see the total time spent on that high-level task.
*   **Error Tracking**: If the code inside the `with` block raises an exception, the span will automatically record the error, making it much easier to debug failed requests in production.

## Tracing Multiple Operations with Named Spans

Great job learning how to use custom spans for individual operations! Now, let's take your tracing skills further by breaking down complex functions into multiple detailed spans.

In real-world applications, a single function often performs several different operations, and you want to see exactly which step is taking the most time or causing issues. Instead of tracing the entire function as one big block, you can create separate spans for each major operation.

Your task is to add two different named spans to trace the distinct operations in the process_order() function:

    Wrap the document reference creation with a span named 'get-document-reference'.
    Wrap the data writing operation with a span named 'write-document'.

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import ConsoleSpanExporter
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient
from google.cloud import firestore

# Configure provider with resource info
provider = TracerProvider(resource=Resource.create({"service.name": "orders-worker"}))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("OrdersWorker")

# Export spans to stdout (prints JSON to console)
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))

# Instrument gRPC before Firestore client
GrpcInstrumentorClient().instrument()
db = firestore.Client()

def process_order():
    # TODO: Wrap the document reference creation below with a span named 'get-document-reference'
    doc_ref = db.collection("orders").document("o-500")
    order_data = {"order_id": "o-500", "amount": 199.99, "status": "pending"}
    # TODO: Wrap the set operation below with a span named 'write-document'
    doc_ref.set(order_data)

if __name__ == "__main__":
    process_order()
    provider.shutdown()
    print("Done")

```
By nesting multiple spans within a single function, you create a hierarchy. In your console output, you will now see these as two distinct events, allowing you to pinpoint exactly where latency occurs.

Here is the updated code with both spans implemented:

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import ConsoleSpanExporter
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient
from google.cloud import firestore

# Configure provider with resource info
provider = TracerProvider(resource=Resource.create({"service.name": "orders-worker"}))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("OrdersWorker")

# Export spans to stdout (prints JSON to console)
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))

# Instrument gRPC before Firestore client
GrpcInstrumentorClient().instrument()
db = firestore.Client()

def process_order():
    # --- TODO FIXED ---
    # 1. Wrap document reference creation
    with tracer.start_as_current_span("get-document-reference"):
        doc_ref = db.collection("orders").document("o-500")
    
    order_data = {"order_id": "o-500", "amount": 199.99, "status": "pending"}
    
    # 2. Wrap the set operation
    with tracer.start_as_current_span("write-document"):
        doc_ref.set(order_data)
    # ------------------

if __name__ == "__main__":
    process_order()
    provider.shutdown()
    print("Done")
```

### Understanding Span Relationships
When you run this, OpenTelemetry creates a "trace context." Even though these are two separate `with` blocks, they are part of the same execution flow.



*   **`get-document-reference`**: This span is usually very fast because it's a local client-side operation (it doesn't actually talk to the database yet).
*   **`write-document`**: This span will take longer because it involves a network request. Because you have `GrpcInstrumentorClient` active, you will actually see a "child" span automatically generated *inside* this one that represents the actual gRPC call to Google's servers.

This granular approach is the "gold standard" for debugging production systems—you'll never have to guess which specific line of code is slowing down your users.

## Fixing Broken OpenTelemetry Tracing for Google Cloud Firestore Operations

You have a Python script that's supposed to trace multiple Google Cloud Firestore operations using OpenTelemetry, but it contains several common tracing setup mistakes that are preventing tracing from working properly. This is your chance to apply all the concepts you've learned and fix a realistic, broken implementation.

Your task is to identify and fix all the OpenTelemetry integration issues in this code:

    Missing imports for essential OpenTelemetry modules
    Incorrect configuration timing and setup order
    Missing function calls for automatic tracing/instrumentation
    Broken spans that don't properly wrap their operations
    Missing cleanup calls that ensure traces are exported

Some problems have TODO comments to guide you, but others look like normal code that you'll need to debug yourself. The script should perform multiple Firestore operations while properly tracing each one with both automatic instrumentation and custom spans.

When you fix all the issues, you'll have a fully functional OpenTelemetry setup that demonstrates mastery of distributed tracing concepts and prepares you to debug similar problems in real applications.

```python
from opentelemetry import trace
from google.cloud import firestore

# TODO: Import the required OpenTelemetry SDK modules for tracing setup

# TODO: Set up the tracer provider with resource info

# TODO: Configure the console span exporter

# TODO: Add the missing instrumentation for gRPC calls

db = firestore.Client()

def process_orders():
    # TODO: Add missing span setup and fix span coverage for each operation
    
    db.collection("orders").document("o-100").set({
        "order_id": "o-100",
        "amount": 45.99,
        "status": "pending"
    })
    
    db.collection("orders").document("o-101").set({
        "order_id": "o-101",
        "amount": 78.50,
        "status": "confirmed"
    })

def check_orders():
    # TODO: Add a span to trace the query operation
    orders_ref = db.collection("orders")
    docs = orders_ref.limit(5).stream()
    items = list(docs)
    return len(items)

if __name__ == "__main__":
    process_orders()
    count = check_orders()
    # TODO: Add proper cleanup to ensure traces are exported
    print(f"Processed orders, found {count} items")
    print("Done")

```

This is a great exercise! Debugging a broken tracing implementation is often harder than writing one from scratch because the order of operations is so sensitive.

To fix this script, we need to ensure the **TracerProvider** is globally registered and the **gRPC instrumentation** is active *before* the Firestore client is even initialized. Otherwise, the automatic tracing won't "hook" into the client.

Here is the fully repaired and functional implementation:

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.instrumentation.grpc import GrpcInstrumentorClient
from google.cloud import firestore

# 1. Set up the tracer provider with resource info (Service Name)
resource = Resource.create({"service.name": "order-processor"})
provider = TracerProvider(resource=resource)
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("OrdersApp")

# 2. Configure the console span exporter with a processor
console_exporter = ConsoleSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))

# 3. CRITICAL: Add instrumentation for gRPC calls BEFORE creating the client
GrpcInstrumentorClient().instrument()

# Now it is safe to create the Firestore client
db = firestore.Client()

def process_orders():
    # 4. Wrap the batch of operations in a high-level span
    with tracer.start_as_current_span("process-multiple-orders"):
        # We can even create sub-spans for individual writes if needed
        db.collection("orders").document("o-100").set({
            "order_id": "o-100",
            "amount": 45.99,
            "status": "pending"
        })
        
        db.collection("orders").document("o-101").set({
            "order_id": "o-101",
            "amount": 78.50,
            "status": "confirmed"
        })

def check_orders():
    # 5. Add a span to trace the query operation
    with tracer.start_as_current_span("query-orders-limit"):
        orders_ref = db.collection("orders")
        docs = orders_ref.limit(5).stream()
        items = list(docs)
        return len(items)

if __name__ == "__main__":
    process_orders()
    count = check_orders()
    
    # 6. Proper cleanup to flush the buffers and ensure traces reach the console
    provider.shutdown()
    
    print(f"Processed orders, found {count} items")
    print("Done")
```



### Key Fixes Explained:

*   **Import Strategy**: We added `TracerProvider`, `SimpleSpanProcessor`, `ConsoleSpanExporter`, and `Resource`. Without these SDK components, the `trace` API has nowhere to send the data.
*   **The "Golden Rule" of Order**: Notice that `GrpcInstrumentorClient().instrument()` is called before `db = firestore.Client()`. If you swap these, the Firestore client uses a standard gRPC library that hasn't been "patched" to record spans.
*   **Span Coverage**: In `process_orders`, we used a single span to group the writes. In `check_orders`, we wrapped the `.stream()` and `list()` conversion, as the network activity happens when the stream is actually consumed.
*   **`provider.shutdown()`**: This is often forgotten. Without it, the Python script might exit while the `SimpleSpanProcessor` is still trying to print the last few spans to the console, leading to missing data.

How would you modify the processor if you were moving this code from your local terminal to a high-traffic production environment?